# Basic Analysis of MTG Card Data
Using oracle card data from Scryfall. One object per unique card, not per printing.

In [2]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

## Load and Filter
Load the raw JSON, filter to physical cards only, and set oracle_id as the index.

In [3]:
cards = pd.read_json('oracle-cards-20260305100222.json')
simplified_cards = cards[cards['digital'] == False].copy()
simplified_cards.set_index('oracle_id', inplace=True)

print(f'{len(simplified_cards)} physical cards loaded')
simplified_cards.info()

34806 physical cards loaded
<class 'pandas.core.frame.DataFrame'>
Index: 34806 entries, 00037840-6089-42ec-8c5c-281f9f474504 to ffff90c3-63c4-4dee-a21d-6b2b113f4f80
Data columns (total 85 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   object               34806 non-null  object        
 1   id                   34806 non-null  object        
 2   multiverse_ids       34806 non-null  object        
 3   mtgo_id              26132 non-null  float64       
 4   tcgplayer_id         31658 non-null  float64       
 5   cardmarket_id        30596 non-null  float64       
 6   name                 34806 non-null  object        
 7   lang                 34806 non-null  object        
 8   released_at          34806 non-null  datetime64[ns]
 9   uri                  34806 non-null  object        
 10  scryfall_uri         34806 non-null  object        
 11  layout               34806 non-null  object        
 12 

## Dtype Casting
Cast string columns, convert cmc to int, and replace non-numeric power/toughness values with sentinel integers.

| Sentinel | Meaning |
|---|---|
| -2 | null / missing |
| -3 | `*` (variable) |
| -4 | `1+*` / `*+1` |
| -5 | `2+*` / `*+2` |
| -6 | `7-*` |
| -99 | unknown catch-all |

In [4]:
# String columns
string_cols = ['power', 'toughness', 'name', 'type_line', 'oracle_text',
               'set', 'set_type', 'watermark', 'rarity', 'printed_text']
for col in string_cols:
    simplified_cards[col] = simplified_cards[col].astype('string')

simplified_cards['cmc'] = simplified_cards['cmc'].astype('int')

# Power / toughness sentinel mapping
def to_int_safe(val):
    sentinel_map = {'*': -3, '1+*': -4, '*+1': -4, '2+*': -5, '*+2': -5, '7-*': -6}
    if pd.isna(val):
        return -2
    if val in sentinel_map:
        return sentinel_map[val]
    try:
        return int(val)
    except (ValueError, TypeError):
        return -99

for col in ['power', 'toughness']:
    simplified_cards[col] = simplified_cards[col].apply(to_int_safe)

print(simplified_cards[['power', 'toughness']].dtypes)
print('Unique power values:', sorted(simplified_cards['power'].unique()))

power        int64
toughness    int64
dtype: object
Unique power values: [-99, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 20, 99]


## Color Analysis
Add a color count column and build per-color boolean masks for filtering.

In [5]:
simplified_cards['color_count'] = simplified_cards['color_identity'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

color_flags = {
    'White': 'W', 'Blue': 'U', 'Black': 'B', 'Red': 'R', 'Green': 'G'
}
for name, code in color_flags.items():
    simplified_cards[f'is_{name.lower()}'] = simplified_cards['color_identity'].apply(
        lambda x: code in x if isinstance(x, list) else False
    )

# Dict of per-color dataframes for convenience
cards_by_color = {
    name: simplified_cards[simplified_cards[f'is_{name.lower()}']]
    for name in color_flags
}

print(simplified_cards['color_count'].value_counts().sort_index())

color_count
0     5849
1    23466
2     4429
3      881
4       22
5      159
Name: count, dtype: int64


## PCA — Dimensionality Reduction
Encode categorical features numerically, scale, then run PCA to find which features carry the most information and which are redundant.

In [6]:
feature_cols = ['cmc', 'rarity', 'type_line', 'colors', 'color_identity', 'set_type', 'power', 'toughness']
df_features = simplified_cards[feature_cols].copy()

# Encode categoricals
le = LabelEncoder()
for col in ['rarity', 'type_line', 'set_type']:
    df_features[col] = le.fit_transform(df_features[col].astype(str))

# Encode color lists as count (length)
for col in ['colors', 'color_identity']:
    df_features[col] = df_features[col].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

# Scale and run PCA
X = StandardScaler().fit_transform(df_features.fillna(0))
pca = PCA()
pca.fit(X)

# Variance explained per component
explained = pd.DataFrame({
    'component': range(1, len(pca.explained_variance_ratio_) + 1),
    'variance_explained': pca.explained_variance_ratio_,
    'cumulative': np.cumsum(pca.explained_variance_ratio_)
})
print(explained)

# Which original features drive each component
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=[f'PC{i+1}' for i in range(len(feature_cols))]
)
print('\nFeature loadings (sorted by PC1):')
print(loadings.abs().sort_values('PC1', ascending=False))

   component  variance_explained  cumulative
0          1            0.301625    0.301625
1          2            0.212427    0.514052
2          3            0.139000    0.653052
3          4            0.125001    0.778053
4          5            0.097332    0.875386
5          6            0.078484    0.953870
6          7            0.025257    0.979127
7          8            0.020873    1.000000

Feature loadings (sorted by PC1):
                     PC1       PC2       PC3       PC4       PC5       PC6  \
colors          0.528497  0.276503  0.165096  0.001382  0.011800  0.311093   
color_identity  0.507345  0.315905  0.131426  0.000944  0.033353  0.385022   
toughness       0.426211  0.510111  0.076901  0.008529  0.049210  0.203595   
power           0.416847  0.515778  0.085348  0.009117  0.057774  0.236551   
set_type        0.229600  0.104032  0.629027  0.011743  0.704863  0.208770   
rarity          0.177623  0.125262  0.683059  0.000377  0.696378  0.035011   
type_line     

In [7]:
for col in loadings.columns:
    top = loadings[col].abs().nlargest(3)
    print(f'\n{col}:')
    print(top)


PC1:
colors            0.528497
color_identity    0.507345
toughness         0.426211
Name: PC1, dtype: float64

PC2:
type_line    0.520572
power        0.515778
toughness    0.510111
Name: PC2, dtype: float64

PC3:
rarity       0.683059
set_type     0.629027
type_line    0.282563
Name: PC3, dtype: float64

PC4:
cmc          0.999697
type_line    0.017564
set_type     0.011743
Name: PC4, dtype: float64

PC5:
set_type     0.704863
rarity       0.696378
type_line    0.105289
Name: PC5, dtype: float64

PC6:
type_line         0.782644
color_identity    0.385022
colors            0.311093
Name: PC6, dtype: float64

PC7:
colors            0.717865
color_identity    0.688753
power             0.091146
Name: PC7, dtype: float64

PC8:
toughness    0.712582
power        0.696594
colors       0.068778
Name: PC8, dtype: float64
